<a href="https://colab.research.google.com/github/kumVij/resume/blob/main/Agentic_Interview_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Agentic AI + Java Integration — Interview Prep Guide

---

## 1. Core Concepts

### What is Agentic AI?
Software where an LLM doesn't just respond to a single prompt, but **plans, decides, and acts** — calling tools/APIs, observing results, and deciding the next step in a loop, with minimal human intervention per step. The key shift from a normal LLM app: the model is given **agency** over what to do next, not just what to say.

### Agent vs a plain LLM call (common opener)
| Plain LLM call | Agent |
|---|---|
| Single prompt → single response | Multi-step loop: reason → act → observe → repeat |
| No tool use, no external state | Calls tools/APIs, reads results, adapts plan |
| Stateless per request | Often maintains memory/context across steps |
| Deterministic pipeline (you control the flow) | Model itself decides control flow (which tool, when to stop) |

### The ReAct Loop (Reason + Act) — foundational pattern, near-guaranteed question
1. **Thought** — model reasons about what to do next.
2. **Action** — model calls a tool (search, DB query, API call).
3. **Observation** — tool result is fed back to the model.
4. Repeat until the model decides it has enough information to give a final answer.

### Key Building Blocks of an Agent
- **LLM** — the reasoning engine (planning, tool selection, response generation).
- **Tools/Functions** — external capabilities the model can invoke (APIs, DB queries, code execution).
- **Memory** — short-term (conversation context) and long-term (vector store, DB) recall.
- **Orchestrator/Planner** — the loop or framework driving reason→act→observe.
- **Guardrails** — validation, human-in-the-loop approval, risk scoring before executing risky actions.

---

## 2. Tool/Function Calling (the mechanism that makes agents "do" things)

Most modern LLM providers support **structured function calling**: you describe available functions (name, description, JSON schema of parameters); the model returns a structured request to call one, instead of free text.

```json
// Tool definition sent to the LLM
{
  "name": "getWeather",
  "description": "Get current weather for a city",
  "parameters": {
    "type": "object",
    "properties": { "city": { "type": "string" } },
    "required": ["city"]
  }
}
```
```json
// Model's response requesting a tool call (not the final answer)
{ "tool_call": { "name": "getWeather", "arguments": { "city": "Bengaluru" } } }
```
Your application code executes the actual function, then sends the **result** back to the model as a new message so it can continue reasoning or produce a final answer.

**Interview point:** the LLM never directly executes code — it only ever emits a structured intent to call a tool. Your backend is always the one actually invoking APIs/DB calls, which is exactly where validation and guardrails belong.

---

## 3. Agentic AI in Java — the Ecosystem

Java doesn't yet have as mature an agent ecosystem as Python (LangChain, LlamaIndex), but it's converging fast around two main options:

### Spring AI
Spring's official abstraction layer over LLM providers (OpenAI, Anthropic, Azure, Ollama, etc.), designed to feel idiomatic to Spring Boot developers — `@Bean`-based config, auto-configuration, and integration with Spring's existing ecosystem (Spring Data, Spring Cloud).
```java
@Service
public class ChatService {

    private final ChatClient chatClient;

    public ChatService(ChatClient.Builder builder) {
        this.chatClient = builder.build();
    }

    public String ask(String question) {
        return chatClient.prompt()
                .user(question)
                .call()
                .content();
    }
}
```

### LangChain4j
A Java port of LangChain's core concepts — chains, agents, tools, memory, RAG — with a more explicit, builder-style API than Spring AI. Popular when you're not already in a Spring context, or want closer parity with LangChain patterns from Python tutorials.

```java
interface Assistant {
    String chat(String userMessage);
}

Assistant assistant = AiServices.builder(Assistant.class)
        .chatLanguageModel(OpenAiChatModel.withApiKey(apiKey))
        .tools(new WeatherTool(), new CalculatorTool())
        .chatMemory(MessageWindowChatMemory.withMaxMessages(10))
        .build();

String response = assistant.chat("What's the weather in Bengaluru, then multiply that temp by 2?");
```

---

## 4. Defining Tools in Java

### Spring AI — annotation-based
```java
@Component
public class WeatherTools {

    @Tool(description = "Get current weather for a given city")
    public String getWeather(String city) {
        // call actual weather API
        return weatherClient.fetch(city);
    }
}

// registered when building the ChatClient:
chatClient.prompt()
    .user("What's the weather in Bengaluru?")
    .tools(weatherTools)
    .call()
    .content();
```

### LangChain4j — annotation-based
```java
class CalculatorTool {
    @Tool("Adds two numbers")
    double add(double a, double b) {
        return a + b;
    }
}
```
Both frameworks introspect the method signature + `@Tool` description to auto-generate the JSON schema sent to the LLM — you rarely hand-write the schema yourself.

---

## 5. RAG (Retrieval-Augmented Generation) in Java

Agents commonly need to ground responses in your own data via a vector store lookup before (or during) reasoning.

```java
// LangChain4j example: embedding + retrieval
EmbeddingModel embeddingModel = new AllMiniLmL6V2EmbeddingModel();
EmbeddingStore<TextSegment> store = new InMemoryEmbeddingStore<>();

// Ingest
List<TextSegment> segments = DocumentSplitters.recursive(300, 0).split(document);
List<Embedding> embeddings = embeddingModel.embedAll(segments).content();
store.addAll(embeddings, segments);

// Retrieve
Embedding queryEmbedding = embeddingModel.embed(userQuery).content();
List<EmbeddingMatch<TextSegment>> relevant = store.findRelevant(queryEmbedding, 5);
```
Common vector store integrations from Java: Pinecone, Weaviate, Milvus, pgvector (via plain JDBC or Spring Data), ChromaDB (via HTTP client — no first-class Java SDK, so typically called via REST).

**Interview point:** RAG isn't itself "agentic" — it's a retrieval step. It becomes part of an agent when the model *decides* whether/when to retrieve, rather than retrieval being a fixed step in every request.

---

## 6. Orchestration Patterns

### Single-agent tool loop
One LLM, a set of tools, loops until done — the simplest agent pattern (shown in section 2).

### Multi-agent / orchestrator pattern
A coordinator agent delegates subtasks to specialized agents (e.g., a "planner" agent breaks work down, a "coder" agent writes code, a "reviewer" agent checks it) — useful when a single prompt/tool-set gets too broad or error-prone.

### Event-driven agent (production pattern)
Instead of a synchronous chat loop, the agent is triggered by external events (webhooks, message queue events) and acts autonomously — very common in Java backend systems.
```java
@Component
public class GithubWebhookListener {

    @PostMapping("/webhook/github")
    public ResponseEntity<Void> handle(@RequestBody GithubEvent event) {
        // idempotency check before processing
        if (processedEvents.contains(event.getDeliveryId())) {
            return ResponseEntity.ok().build();
        }
        agentOrchestrator.evaluate(event); // agent reasons about the event, may call tools
        return ResponseEntity.accepted().build();
    }
}
```

---

## 7. Memory & State

- **Short-term (conversation) memory**: sliding window of recent messages, kept in memory or a session store (Redis).
- **Long-term memory**: embeddings stored in a vector DB, retrieved as needed (semantic memory), or structured facts stored in a relational DB.
- **Working memory / scratchpad**: intermediate reasoning state during a single multi-step task, often just the accumulating message history sent back to the LLM each loop iteration.

```java
ChatMemory memory = MessageWindowChatMemory.withMaxMessages(20);
memory.add(UserMessage.from(userInput));
Response<AiMessage> response = model.generate(memory.messages());
memory.add(response.content());
```

---

## 8. Guardrails & Production Concerns (what separates "demo agent" from "production agent")

- **Human-in-the-loop approval gates** for high-risk or irreversible actions (e.g., deploying code, sending external emails, spending money) — the agent proposes an action, a human confirms before execution.
- **Risk scoring** — classify a proposed tool call's potential impact before auto-executing vs escalating for approval.
- **Idempotency** — agents triggered by webhooks/events must handle duplicate deliveries safely (store processed event IDs, use idempotency keys).
- **Timeouts and loop limits** — cap the number of reasoning steps to prevent infinite tool-call loops (a real, common failure mode called a "cascade loop").
- **Structured output validation** — validate the LLM's tool-call JSON against your schema before executing; never trust it blindly.
- **Observability** — log every reasoning step, tool call, and result for debugging and auditing — agent behavior is much harder to reproduce than deterministic code.

```java
@Service
public class RiskScorer {
    public RiskLevel evaluate(ProposedAction action) {
        if (action.isDestructive() || action.getEstimatedImpact() > THRESHOLD) {
            return RiskLevel.REQUIRES_APPROVAL;
        }
        return RiskLevel.AUTO_APPROVE;
    }
}
```

---

## 9. Common Coding Interview Exercises

**a) Implement a basic tool-calling loop (concept, framework-agnostic)**
```java
public String runAgentLoop(String userInput, List<Tool> tools, int maxSteps) {
    List<Message> history = new ArrayList<>(List.of(new UserMessage(userInput)));

    for (int step = 0; step < maxSteps; step++) {
        LlmResponse response = llmClient.call(history, tools);

        if (response.isFinalAnswer()) {
            return response.getContent();
        }

        ToolCall call = response.getToolCall();
        Object result = executeTool(call, tools);       // your code actually runs it
        history.add(new ToolResultMessage(call.getName(), result));
    }
    throw new AgentLoopLimitExceededException();          // guardrail: prevent infinite loops
}
```

**b) Idempotent webhook-triggered agent handler** — see section 6 example; commonly extended with a Redis/DB-backed dedup store instead of an in-memory set.

**c) Simple RAG-backed question answering** — see section 5; often asked to explain the flow verbally more than write full code (embedding → similarity search → prompt augmentation → generation).

---

## 10. Frequently Asked Conceptual Questions

1. **What makes an application "agentic" rather than just an LLM wrapper?** The model itself decides the sequence of actions/tools to use based on intermediate results, rather than following a fixed, pre-programmed pipeline.
2. **How does tool/function calling actually work under the hood?** The LLM is given tool schemas as part of the prompt/API call; it returns structured JSON describing an intended call rather than plain text; your application executes it and returns the result as a new message.
3. **How would you integrate an LLM agent into an existing Java/Spring Boot backend?** Typically as a service layer (Spring AI's `ChatClient` or LangChain4j's `AiServices`) invoked from a controller or triggered by an event listener, with tools implemented as regular Spring-managed beans/methods.
4. **What's the difference between RAG and an agent?** RAG is a retrieval step to ground a response in relevant data; an agent is a broader control-flow pattern where the model decides what actions (including, optionally, retrieval) to take. RAG can be one tool available to an agent.
5. **How do you prevent an agent from getting stuck in a loop or taking a costly wrong action?** Step/time limits, risk scoring before executing sensitive actions, human-in-the-loop approval gates, and strict validation of tool-call arguments before execution.
6. **Why might you avoid a heavy framework like LangChain4j and call the LLM API directly with `httpx`/`RestTemplate`/`WebClient`?** More control over request/response handling, fewer dependency-version conflicts, easier debugging — a legitimate production trade-off, not just "not knowing the framework."
7. **How do you handle non-deterministic LLM output in a production Java system?** Strict output schemas (JSON mode / structured outputs), server-side validation before acting on the response, retries with clarifying prompts on invalid output, and comprehensive logging since behavior isn't perfectly reproducible.
8. **What's the role of embeddings in an agentic system?** Convert text into vectors for semantic similarity search — used for long-term memory retrieval and RAG grounding, not for the agent's step-by-step reasoning itself.
9. **How would you test an agentic system, given LLM output isn't deterministic?** Test the deterministic parts in isolation (tool implementations, guardrail logic, orchestration loop with mocked LLM responses); for the LLM-facing parts, use golden-set evaluation (a fixed set of inputs with expected tool calls/behaviors) rather than exact-output assertions.
10. **What's the difference between Spring AI and LangChain4j?** Spring AI is Spring's own abstraction, tightly integrated with Spring Boot conventions (auto-config, DI) — natural fit if you're already in a Spring shop. LangChain4j is more framework-agnostic with a more explicit builder API and closer conceptual parity to Python's LangChain.

---

## 11. Best Practices

- **Keep the LLM's job narrow: decide, don't execute.** All actual execution (API calls, DB writes) happens in your own validated code.
- **Add a human-in-the-loop gate for irreversible or high-impact actions** — don't fully automate anything that's expensive to undo.
- **Cap reasoning steps and set timeouts** to guard against runaway loops.
- **Validate every tool-call argument against a schema** before executing — never trust LLM output blindly, even in "structured output" mode.
- **Make event-driven agents idempotent** — webhooks and message queues can and will redeliver events.
- **Log every step** (thought, tool call, result) for debugging — agentic systems are much harder to reason about after the fact than deterministic code.
- **Separate the reasoning loop from tool implementations** so tools can be unit-tested independently of the LLM.
- **Prefer structured/function-calling APIs over parsing free text** for reliability — free-text parsing of "which tool to call" is brittle and a common source of bugs.
- **Version and pin your model** — silent model updates from a provider can change agent behavior; pin versions in production and re-validate on upgrade.

---

## 12. Quick Self-Check
Before an interview, you should be able to, without notes:
- Explain the ReAct loop (reason → act → observe) end to end.
- Explain how function/tool calling actually works between your app and the LLM API.
- Sketch how you'd wire an LLM agent into a Spring Boot service, including where tools and guardrails live.
- Explain the difference between RAG and an agent, and where RAG fits as one tool an agent might use.
- Explain at least two concrete guardrails you'd add before letting an agent take real-world actions (approval gates, risk scoring, idempotency, step limits).
- Explain how you'd test a system whose core component (the LLM) is non-deterministic.